In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/)
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import torch  # import torch first to avoid circular import
from kaggle_secrets import UserSecretsClient
import wandb

# Get API key from Kaggle Secrets
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

# Login to wandb
wandb.login(key=wandb_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [3]:
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
import lightgbm as lgb
from sentence_transformers import CrossEncoder, InputExample
import wandb

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# MILESTONE 5

Setup (Run Before Attempting Questions) : 

Use the following two fine-tuned sequence classification checkpoints:

DeBERTa: microsoft/deberta-v3-small (fine-tuned checkpoint)

RoBERTa: roberta-base (fine-tuned checkpoint)

Label Mapping
The models output logits for five labels corresponding to the answer options:

Label ID         Option

0                       A

1                       B

2                       C

3                       D

4                       E

# Load the fine-tuned DeBERTa and RoBERTa models.

# For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

# Question 1:

# Which answer option receives the highest probability from the DeBERTa model, and what is that probability?

(answer format : eg - A, probability of A)

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def perform_inference(model_path, tokenizer_path, prompt, options_dict=None):
    """
    Loads model and tokenizer, tokenizes input, runs inference, and returns softmax probabilities.
    """
    print(f"\nLoading model and tokenizer from: {model_path}")
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    # Load as Sequence Classification model with 5 output labels (for options A, B, C, D, E)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, 
        num_labels=5, 
        use_safetensors=True
    )
    model.eval()
    
    # Format input:
    # 1. Single prompt format: just the prompt text
    # 2. Concat format: prompt + options list (e.g. prompt [SEP] A: optA [SEP] B: optB ...)
    if options_dict:
        # Example of concatenated format if your model was trained on both question + choices:
        input_text = f"Question: {prompt} Options: " + " ".join([f"{k}) {v}" for k, v in options_dict.items()])
    else:
        input_text = prompt

    print(f"Input text for model: {input_text}")
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1).squeeze().numpy()
        
    return probs

def main():
    # 1. Load row 25 of the dataset
    # Set to 'train.csv' or 'test.csv' depending on the dataset of interest.
    df_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv" 
    print(f"Loading dataset: {df_path}")
    df = pd.read_csv(df_path)
    
    row_idx = 25
    row = df.iloc[row_idx]
    prompt = str(row['prompt'])
    options_dict = {
        'A': str(row['A']),
        'B': str(row['B']),
        'C': str(row['C']),
        'D': str(row['D']),
        'E': str(row['E'])
    }
    
    print(f"\nRow Index: {row_idx}")
    print(f"Prompt: {prompt}")
    for k, v in options_dict.items():
        print(f"Option {k}: {v}")
    if 'answer' in row:
        print(f"Correct Answer (Ground Truth): {row['answer']}")
        
    options = ['A', 'B', 'C', 'D', 'E']
    
    # 2. DeBERTa Model
    # Replace with your local fine-tuned checkpoint directory path if saved locally
    deberta_path = "microsoft/deberta-v3-small" 
    try:
        deberta_probs = perform_inference(deberta_path, deberta_path, prompt)
        print("\n--- DeBERTa Inference Results ---")
        for opt, prob in zip(options, deberta_probs):
            print(f"Option {opt} Probability: {prob:.6f}")
        max_idx = np.argmax(deberta_probs)
        print(f"Highest Probability Option: {options[max_idx]} (Probability: {deberta_probs[max_idx]:.6f})")
    except Exception as e:
        print(f"Error running DeBERTa: {e}")

    # 3. RoBERTa Model
    # Replace with your local fine-tuned checkpoint directory path if saved locally
    roberta_path = "roberta-base"
    try:
        roberta_probs = perform_inference(roberta_path, roberta_path, prompt)
        print("\n--- RoBERTa Inference Results ---")
        for opt, prob in zip(options, roberta_probs):
            print(f"Option {opt} Probability: {prob:.6f}")
        max_idx = np.argmax(roberta_probs)
        print(f"Highest Probability Option: {options[max_idx]} (Probability: {roberta_probs[max_idx]:.6f})")
    except Exception as e:
        print(f"Error running RoBERTa: {e}")

if __name__ == "__main__":
    main()

Loading dataset: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv

Row Index: 25
Prompt: Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully.
Option A: Hesse's principle of transfer is a concept in biology that explains the transfer of genetic information from one generation to another.
Option B: Hesse's principle of transfer is a concept in chemistry that explains the transfer of electrons between atoms in a chemical reaction.
Option C: Hesse's principle of transfer is a concept in physics that explains the transfer of energy from one object to another.
Option D: Hesse's principle of transfer is a concept in economics that explains the transfer of wealth from one individual to another.
Option E: Hesse's principle of transfer is a concept in geometry that states that if the points of the projective line P1 are depicted by a rational normal curve in Pn, then the group of the projective transformations of Pn that preserve the curve is is

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Input text for model: Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully.

--- RoBERTa Inference Results ---
Option A Probability: 0.207087
Option B Probability: 0.180497
Option C Probability: 0.202305
Option D Probability: 0.211687
Option E Probability: 0.198425
Highest Probability Option: D (Probability: 0.211687)


# Using the same sample (row index 25), average the class probabilities from both models.

# Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

# Question 2:

# Which answer option receives the highest averaged probability after simple probability ensembling?

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def perform_inference(model_path, tokenizer_path, prompt, options_dict=None):
    """
    Loads model and tokenizer, tokenizes input, runs inference, and returns softmax probabilities.
    """
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, 
        num_labels=5, 
        use_safetensors=True
    )
    model.eval()
    
    if options_dict:
        input_text = f"Question: {prompt} Options: " + " ".join([f"{k}) {v}" for k, v in options_dict.items()])
    else:
        input_text = prompt

    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1).squeeze().numpy()
        
    return probs

def main():
    # 1. Load row 25 of the dataset
    # Set to 'train.csv' or 'test.csv' depending on the dataset of interest.
    df_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv" 
    print(f"Loading dataset: {df_path}")
    df = pd.read_csv(df_path)
    
    row_idx = 25
    row = df.iloc[row_idx]
    prompt = str(row['prompt'])
    options_dict = {
        'A': str(row['A']),
        'B': str(row['B']),
        'C': str(row['C']),
        'D': str(row['D']),
        'E': str(row['E'])
    }
    
    options = ['A', 'B', 'C', 'D', 'E']
    
    # 2. DeBERTa Model
    # Replace with local fine-tuned directory path if checkpoints are stored locally
    deberta_path = "microsoft/deberta-v3-small" 
    try:
        deberta_probs = perform_inference(deberta_path, deberta_path, prompt)
    except Exception as e:
        print(f"Error loading DeBERTa: {e}")
        return

    # 3. RoBERTa Model
    # Replace with local fine-tuned directory path if checkpoints are stored locally
    roberta_path = "roberta-base"
    try:
        roberta_probs = perform_inference(roberta_path, roberta_path, prompt)
    except Exception as e:
        print(f"Error loading RoBERTa: {e}")
        return
    
    # 4. Average the probabilities
    avg_probs = (deberta_probs + roberta_probs) / 2.0
    
    print("\n--- Individual & Averaged Probabilities ---")
    for opt, d_p, r_p, avg_p in zip(options, deberta_probs, roberta_probs, avg_probs):
        print(f"Option {opt} | DeBERTa Prob: {d_p:.6f} | RoBERTa Prob: {r_p:.6f} | Averaged Prob: {avg_p:.6f}")
        
    max_idx = np.argmax(avg_probs)
    print(f"\nHighest Averaged Probability Option: {options[max_idx]} (Averaged Probability: {avg_probs[max_idx]:.6f})")

if __name__ == "__main__":
    main()

Loading dataset: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Individual & Averaged Probabilities ---
Option A | DeBERTa Prob: 0.215698 | RoBERTa Prob: 0.173198 | Averaged Prob: 0.194448
Option B | DeBERTa Prob: 0.211304 | RoBERTa Prob: 0.205890 | Averaged Prob: 0.208597
Option C | DeBERTa Prob: 0.177856 | RoBERTa Prob: 0.200066 | Averaged Prob: 0.188961
Option D | DeBERTa Prob: 0.184937 | RoBERTa Prob: 0.193814 | Averaged Prob: 0.189375
Option E | DeBERTa Prob: 0.210327 | RoBERTa Prob: 0.227032 | Averaged Prob: 0.218679

Highest Averaged Probability Option: E (Averaged Probability: 0.218679)


Apply weighted probability averaging after Softmax using the following weights:

DeBERTa: 0.70

RoBERTa: 0.30

Compute:

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

# Question 3:

# Which answer option is ranked first after weighted ensembling?

In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def perform_inference(model_path, tokenizer_path, prompt, options_dict=None):
    """
    Loads model and tokenizer, tokenizes input, runs inference, and returns softmax probabilities.
    """
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, 
        num_labels=5, 
        use_safetensors=True
    )
    model.eval()
    
    if options_dict:
        input_text = f"Question: {prompt} Options: " + " ".join([f"{k}) {v}" for k, v in options_dict.items()])
    else:
        input_text = prompt

    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1).squeeze().numpy()
        
    return probs

def main():
    # 1. Load row 25 of the dataset
    df_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv" 
    print(f"Loading dataset: {df_path}")
    df = pd.read_csv(df_path)
    
    row_idx = 25
    row = df.iloc[row_idx]
    prompt = str(row['prompt'])
    options_dict = {
        'A': str(row['A']),
        'B': str(row['B']),
        'C': str(row['C']),
        'D': str(row['D']),
        'E': str(row['E'])
    }
    
    options = ['A', 'B', 'C', 'D', 'E']
    
    # 2. DeBERTa Model
    deberta_path = "microsoft/deberta-v3-small" 
    try:
        deberta_probs = perform_inference(deberta_path, deberta_path, prompt)
    except Exception as e:
        print(f"Error loading DeBERTa: {e}")
        return

    # 3. RoBERTa Model
    roberta_path = "roberta-base"
    try:
        roberta_probs = perform_inference(roberta_path, roberta_path, prompt)
    except Exception as e:
        print(f"Error loading RoBERTa: {e}")
        return
    
    # 4. Weighted probability averaging
    w_deberta = 0.70
    w_roberta = 0.30
    weighted_probs = (w_deberta * deberta_probs) + (w_roberta * roberta_probs)
    
    print("\n--- Individual & Weighted Probabilities ---")
    for opt, d_p, r_p, w_p in zip(options, deberta_probs, roberta_probs, weighted_probs):
        print(f"Option {opt} | DeBERTa Prob: {d_p:.6f} | RoBERTa Prob: {r_p:.6f} | Weighted Prob: {w_p:.6f}")
        
    max_idx = np.argmax(weighted_probs)
    print(f"\nHighest Weighted Probability Option (Ranked First): {options[max_idx]} (Weighted Probability: {weighted_probs[max_idx]:.6f})")

if __name__ == "__main__":
    main()

Loading dataset: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Individual & Weighted Probabilities ---
Option A | DeBERTa Prob: 0.246216 | RoBERTa Prob: 0.204135 | Weighted Prob: 0.233604
Option B | DeBERTa Prob: 0.192871 | RoBERTa Prob: 0.218342 | Weighted Prob: 0.200512
Option C | DeBERTa Prob: 0.211304 | RoBERTa Prob: 0.184836 | Weighted Prob: 0.203400
Option D | DeBERTa Prob: 0.164795 | RoBERTa Prob: 0.163881 | Weighted Prob: 0.164582
Option E | DeBERTa Prob: 0.184814 | RoBERTa Prob: 0.228807 | Weighted Prob: 0.198037

Highest Weighted Probability Option (Ranked First): A (Weighted Probability: 0.233604)


Using the weighted ensemble probabilities from Q3, rank all five answer options.

Write the final prediction exactly in Kaggle submission format.

# Question 4:

# What is the Top-3 prediction string for row index 25?

Example : C A E



In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def perform_inference(model_path, tokenizer_path, prompt, options_dict=None):
    """
    Loads model and tokenizer, tokenizes input, runs inference, and returns softmax probabilities.
    """
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, 
        num_labels=5, 
        use_safetensors=True
    )
    model.eval()
    
    if options_dict:
        input_text = f"Question: {prompt} Options: " + " ".join([f"{k}) {v}" for k, v in options_dict.items()])
    else:
        input_text = prompt

    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1).squeeze().numpy()
        
    return probs

def main():
    # 1. Load row 25 of the dataset
    df_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv" 
    print(f"Loading dataset: {df_path}")
    df = pd.read_csv(df_path)
    
    row_idx = 25
    row = df.iloc[row_idx]
    prompt = str(row['prompt'])
    options_dict = {
        'A': str(row['A']),
        'B': str(row['B']),
        'C': str(row['C']),
        'D': str(row['D']),
        'E': str(row['E'])
    }
    
    options = ['A', 'B', 'C', 'D', 'E']
    
    # 2. DeBERTa Model
    deberta_path = "microsoft/deberta-v3-small" 
    try:
        deberta_probs = perform_inference(deberta_path, deberta_path, prompt)
    except Exception as e:
        print(f"Error loading DeBERTa: {e}")
        return

    # 3. RoBERTa Model
    roberta_path = "roberta-base"
    try:
        roberta_probs = perform_inference(roberta_path, roberta_path, prompt)
    except Exception as e:
        print(f"Error loading RoBERTa: {e}")
        return
    
    # 4. Weighted probability averaging (Q3 settings)
    w_deberta = 0.70
    w_roberta = 0.30
    weighted_probs = (w_deberta * deberta_probs) + (w_roberta * roberta_probs)
    
    # 5. Sort probabilities in descending order and format the Top-3 prediction string
    sorted_indices = np.argsort(weighted_probs)[::-1]
    top_3_predictions = [options[idx] for idx in sorted_indices[:3]]
    prediction_string = " ".join(top_3_predictions)
    
    print("\n--- Weighted Probabilities ---")
    for opt, prob in zip(options, weighted_probs):
        print(f"Option {opt} | Weighted Prob: {prob:.6f}")
        
    print(f"\nAll Ranked Options: {[options[idx] for idx in sorted_indices]}")
    print(f"Top-3 Prediction String: {prediction_string}")

if __name__ == "__main__":
    main()

Loading dataset: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Weighted Probabilities ---
Option A | Weighted Prob: 0.209331
Option B | Weighted Prob: 0.171889
Option C | Weighted Prob: 0.201256
Option D | Weighted Prob: 0.253574
Option E | Weighted Prob: 0.164024

All Ranked Options: ['D', 'A', 'C', 'B', 'E']
Top-3 Prediction String: D A C


Run the weighted ensemble pipeline on every row of test.csv.

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

# Question 5:

# Exactly how many prediction rows are present in the generated file (excluding the header)?



In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

def perform_inference_batch(model_path, tokenizer_path, prompts, batch_size=32):
    """
    Loads model and tokenizer, tokenizes input prompts in batches, 
    runs inference, and returns softmax probabilities.
    """
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    # Ensure pad token is defined for batch tokenization
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, 
        num_labels=5, 
        use_safetensors=True
    )
    model.eval()
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    all_probs = []
    
    # Process in batches
    for i in tqdm(range(0, len(prompts), batch_size), desc=f"Running inference for {model_path}"):
        batch_prompts = prompts[i:i+batch_size]
        inputs = tokenizer(batch_prompts, return_tensors="pt", truncation=True, max_length=512, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits.cpu()
            probs = F.softmax(logits, dim=-1).numpy()
            all_probs.append(probs)
            
    return np.concatenate(all_probs, axis=0)

def main():
    test_df_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv" 
    print(f"Loading dataset: {test_df_path}")
    test_df = pd.read_csv(test_df_path)
    
    prompts = test_df['prompt'].astype(str).tolist()
    options = ['A', 'B', 'C', 'D', 'E']
    
    # 1. DeBERTa Model Inference
    deberta_path = "microsoft/deberta-v3-small" 
    try:
        deberta_probs = perform_inference_batch(deberta_path, deberta_path, prompts)
    except Exception as e:
        print(f"Error loading/running DeBERTa: {e}")
        return

    # 2. RoBERTa Model Inference
    roberta_path = "roberta-base"
    try:
        roberta_probs = perform_inference_batch(roberta_path, roberta_path, prompts)
    except Exception as e:
        print(f"Error loading/running RoBERTa: {e}")
        return
    
    # 3. Weighted Average (DeBERTa: 0.70, RoBERTa: 0.30)
    w_deberta = 0.70
    w_roberta = 0.30
    weighted_probs = (w_deberta * deberta_probs) + (w_roberta * roberta_probs)
    
    # 4. Generate Top-3 predictions
    predictions = []
    for i in range(len(weighted_probs)):
        sorted_indices = np.argsort(weighted_probs[i])[::-1]
        top_3 = [options[idx] for idx in sorted_indices[:3]]
        predictions.append(" ".join(top_3))
        
    # 5. Save to submission.csv
    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'prediction': predictions
    })
    
    output_path = "submission.csv"
    submission_df.to_csv(output_path, index=False)
    print(f"\nSaved predictions to {output_path}")
    
    # 6. Print the number of rows excluding the header
    print(f"Number of prediction rows in the generated file (excluding header): {len(submission_df)}")

if __name__ == "__main__":
    main()

Loading dataset: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Running inference for roberta-base: 100%|██████████| 63/63 [00:04<00:00, 13.50it/s]



Saved predictions to submission.csv
Number of prediction rows in the generated file (excluding header): 2000


For the first 50 rows of test.csv, create two versions of every prompt:

1.Original prompt

2.Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"

Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

# Question 6:

# How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?

In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

def main():
    test_df_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
    print(f"Loading dataset: {test_df_path}")
    test_df = pd.read_csv(test_df_path).head(50)
    
    # Load tokenizer and model
    model_name = "microsoft/deberta-v3-small"
    print(f"Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=5,
        use_safetensors=True
    )
    model.eval()
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    options = ['A', 'B', 'C', 'D', 'E']
    different_predictions_count = 0
    
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Running TTA Inference"):
        orig_prompt = str(row['prompt'])
        aug_prompt = "Answer the following multiple-choice question carefully: " + orig_prompt
        
        # 1. Original prompt inference
        inputs_orig = tokenizer(orig_prompt, return_tensors="pt", truncation=True, max_length=512)
        inputs_orig = {k: v.to(device) for k, v in inputs_orig.items()}
        with torch.no_grad():
            outputs_orig = model(**inputs_orig)
            probs_orig = F.softmax(outputs_orig.logits, dim=-1).squeeze().cpu().numpy()
            
        # 2. Augmented prompt inference
        inputs_aug = tokenizer(aug_prompt, return_tensors="pt", truncation=True, max_length=512)
        inputs_aug = {k: v.to(device) for k, v in inputs_aug.items()}
        with torch.no_grad():
            outputs_aug = model(**inputs_aug)
            probs_aug = F.softmax(outputs_aug.logits, dim=-1).squeeze().cpu().numpy()
            
        # 3. Average the probabilities
        probs_tta = (probs_orig + probs_aug) / 2.0
        
        # 4. Compare Top-1 predictions
        top1_orig = np.argmax(probs_orig)
        top1_tta = np.argmax(probs_tta)
        
        if top1_orig != top1_tta:
            different_predictions_count += 1
            print(f"\nRow {idx} changed prediction:")
            print(f"  Original Top-1: {options[top1_orig]} (probs: {probs_orig})")
            print(f"  TTA Top-1:      {options[top1_tta]} (probs: {probs_tta})")
            
    print("\n==================================================")
    print(f"Number of rows with different Top-1 predictions: {different_predictions_count}")
    print("==================================================")

if __name__ == "__main__":
    main()

Loading dataset: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
Loading microsoft/deberta-v3-small...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         


Row 1 changed prediction:
  Original Top-1: B (probs: [0.1611 0.2285 0.1871 0.2252 0.1978])
  TTA Top-1:      D (probs: [0.187  0.1938 0.1831 0.2654 0.1704])

Row 4 changed prediction:
  Original Top-1: A (probs: [0.2448 0.1566 0.1597 0.2222 0.2167])
  TTA Top-1:      D (probs: [0.2349 0.158  0.1785 0.2496 0.1792])

Row 6 changed prediction:
  Original Top-1: B (probs: [0.1859 0.228  0.2122 0.1627 0.211 ])
  TTA Top-1:      A (probs: [0.2062 0.1992 0.197  0.2046 0.1931])

Row 9 changed prediction:
  Original Top-1: B (probs: [0.1687 0.2404 0.2057 0.1876 0.1976])
  TTA Top-1:      D (probs: [0.1904 0.2053 0.1893 0.217  0.198 ])


Running TTA Inference:  40%|████      | 20/50 [00:00<00:00, 38.80it/s]

...


Row 31 changed prediction:
  Original Top-1: E (probs: [0.2241 0.168  0.1646 0.2146 0.2288])
  TTA Top-1:      D (probs: [0.2188 0.1578 0.1968 0.2255 0.2012])

Row 32 changed prediction:
  Original Top-1: E (probs: [0.1995 0.1683 0.16   0.2236 0.2484])
  TTA Top-1:      D (probs: [0.2041 0.1658 0.1831 0.2427 0.2043])

Row 33 changed prediction:
  Original Top-1: A (probs: [0.2148 0.1691 0.2128 0.1918 0.2115])
  TTA Top-1:      D (probs: [0.2124 0.1637 0.1951 0.249  0.1798])

Row 34 changed prediction:
  Original Top-1: C (probs: [0.1868 0.1946 0.2178 0.2002 0.2006])
  TTA Top-1:      D (probs: [0.1989 0.1768 0.1962 0.2445 0.1838])

Row 36 changed prediction:
  Original Top-1: B (probs: [0.1925 0.2534 0.1381 0.207  0.2091])
  TTA Top-1:      D (probs: [0.2014 0.1997 0.1566 0.26   0.1824])

Row 38 changed prediction:
  Original Top-1: A (probs: [0.2264 0.2026 0.1531 0.2255 0.1924])
  TTA Top-1:      D (probs: [0.2161 0.2078 0.1649 0.2395 0.1718])


Running TTA Inference:  98%|█████████▊| 49/50 [00:01<00:00, 42.04it/s]


Row 40 changed prediction:
  Original Top-1: E (probs: [0.2089 0.1794 0.148  0.224  0.2399])
  TTA Top-1:      D (probs: [0.207  0.1772 0.1672 0.227  0.2214])

Row 44 changed prediction:
  Original Top-1: A (probs: [0.23   0.1921 0.1664 0.2225 0.1891])
  TTA Top-1:      D (probs: [0.2278 0.1792 0.1836 0.2456 0.1638])

Row 46 changed prediction:
  Original Top-1: B (probs: [0.2128 0.2379 0.1832 0.1838 0.1821])
  TTA Top-1:      D (probs: [0.2094 0.2128 0.1843 0.2205 0.173 ])

Row 48 changed prediction:
  Original Top-1: C (probs: [0.1923 0.1766 0.2385 0.2039 0.1887])
  TTA Top-1:      D (probs: [0.2085 0.1649 0.2073 0.2544 0.165 ])


Running TTA Inference: 100%|██████████| 50/50 [00:01<00:00, 39.38it/s]



Number of rows with different Top-1 predictions: 22


Process the first 100 rows of test.csv. And compare the Top-1 prediction from:

1. DeBERTa

2. Weighted Ensemble

# Question 7:

# How many rows have different Top-1 predictions?



In [8]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

def main():
    test_df_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
    print(f"Loading dataset: {test_df_path}")
    test_df = pd.read_csv(test_df_path).head(100)
    
    prompts = test_df['prompt'].astype(str).tolist()
    options = ['A', 'B', 'C', 'D', 'E']
    
    # 1. Load DeBERTa
    model_deberta_name = "microsoft/deberta-v3-small"
    print(f"Loading {model_deberta_name}...")
    tokenizer_deberta = AutoTokenizer.from_pretrained(model_deberta_name)
    model_deberta = AutoModelForSequenceClassification.from_pretrained(
        model_deberta_name,
        num_labels=5,
        use_safetensors=True
    )
    model_deberta.eval()
    
    # 2. Load RoBERTa
    model_roberta_name = "roberta-base"
    print(f"Loading {model_roberta_name}...")
    tokenizer_roberta = AutoTokenizer.from_pretrained(model_roberta_name)
    model_roberta = AutoModelForSequenceClassification.from_pretrained(
        model_roberta_name,
        num_labels=5,
        use_safetensors=True
    )
    model_roberta.eval()
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_deberta = model_deberta.to(device)
    model_roberta = model_roberta.to(device)
    
    different_predictions_count = 0
    
    # Inference loop
    for idx in tqdm(range(len(test_df)), desc="Comparing DeBERTa vs. Weighted Ensemble"):
        prompt = prompts[idx]
        
        # DeBERTa inference
        inputs_deberta = tokenizer_deberta(prompt, return_tensors="pt", truncation=True, max_length=512)
        inputs_deberta = {k: v.to(device) for k, v in inputs_deberta.items()}
        with torch.no_grad():
            outputs_d = model_deberta(**inputs_deberta)
            probs_deberta = F.softmax(outputs_d.logits, dim=-1).squeeze().cpu().numpy()
            
        # RoBERTa inference
        inputs_roberta = tokenizer_roberta(prompt, return_tensors="pt", truncation=True, max_length=512)
        inputs_roberta = {k: v.to(device) for k, v in inputs_roberta.items()}
        with torch.no_grad():
            outputs_r = model_roberta(**inputs_roberta)
            probs_roberta = F.softmax(outputs_r.logits, dim=-1).squeeze().cpu().numpy()
            
        # Weighted Ensemble calculation (0.7 * DeBERTa + 0.3 * RoBERTa)
        w_deberta = 0.70
        w_roberta = 0.30
        probs_ensemble = (w_deberta * probs_deberta) + (w_roberta * probs_roberta)
        
        top1_deberta = np.argmax(probs_deberta)
        top1_ensemble = np.argmax(probs_ensemble)
        
        if top1_deberta != top1_ensemble:
            different_predictions_count += 1
            print(f"\nRow {idx} changed prediction:")
            print(f"  DeBERTa Top-1:       {options[top1_deberta]} (probs: {probs_deberta})")
            print(f"  Weighted Ens Top-1:  {options[top1_ensemble]} (probs: {probs_ensemble})")
            
    print("\n==================================================")
    print(f"Number of rows with different Top-1 predictions: {different_predictions_count}")
    print("==================================================")

if __name__ == "__main__":
    main()

Loading dataset: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
Loading microsoft/deberta-v3-small...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         

Loading roberta-base...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...


Row 50 changed prediction:
  DeBERTa Top-1:       A (probs: [0.2181 0.2134 0.1655 0.2069 0.196 ])
  Weighted Ens Top-1:  D (probs: [0.206442   0.20655927 0.1748268  0.20725279 0.20517549])

Row 52 changed prediction:
  DeBERTa Top-1:       B (probs: [0.1917 0.217  0.1743 0.2142 0.2028])
  Weighted Ens Top-1:  D (probs: [0.18766253 0.20925224 0.18124898 0.21197383 0.21005774])

Row 53 changed prediction:
  DeBERTa Top-1:       D (probs: [0.2085 0.1741 0.1981 0.2112 0.208 ])
  Weighted Ens Top-1:  E (probs: [0.20011292 0.17878342 0.1975452  0.2096936  0.21387708])


Comparing DeBERTa vs. Weighted Ensemble:  92%|█████████▏| 92/100 [00:01<00:00, 46.16it/s]


Row 83 changed prediction:
  DeBERTa Top-1:       D (probs: [0.1958 0.1855 0.1918 0.2169 0.21  ])
  Weighted Ens Top-1:  E (probs: [0.19073021 0.18688238 0.19389272 0.2142353  0.21433264])

Row 92 changed prediction:
  DeBERTa Top-1:       B (probs: [0.1696 0.2136 0.2124 0.1958 0.2087])
  Weighted Ens Top-1:  E (probs: [0.17208925 0.20646054 0.2084891  0.1993661  0.21372929])


Comparing DeBERTa vs. Weighted Ensemble: 100%|██████████| 100/100 [00:02<00:00, 46.76it/s]



Row 93 changed prediction:
  DeBERTa Top-1:       A (probs: [0.2329 0.1748 0.16   0.2085 0.2236])
  Weighted Ens Top-1:  E (probs: [0.21731699 0.17945582 0.17076516 0.20803833 0.22455801])

Row 96 changed prediction:
  DeBERTa Top-1:       D (probs: [0.1852 0.2118 0.1686 0.2188 0.2158])
  Weighted Ens Top-1:  E (probs: [0.18301459 0.20524657 0.17742512 0.21501473 0.21961641])

Row 98 changed prediction:
  DeBERTa Top-1:       A (probs: [0.253  0.1888 0.1365 0.2418 0.18  ])
  Weighted Ens Top-1:  D (probs: [0.23061283 0.18907729 0.15470406 0.23176727 0.19415595])

Number of rows with different Top-1 predictions: 13


For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

# Question 8:

# How many rows have a positive confidence gain (greater than 0)?

In [9]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

def main():
    test_df_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
    print(f"Loading dataset: {test_df_path}")
    test_df = pd.read_csv(test_df_path).head(100)
    
    prompts = test_df['prompt'].astype(str).tolist()
    
    # 1. Load DeBERTa
    model_deberta_name = "microsoft/deberta-v3-small"
    print(f"Loading {model_deberta_name}...")
    tokenizer_deberta = AutoTokenizer.from_pretrained(model_deberta_name)
    model_deberta = AutoModelForSequenceClassification.from_pretrained(
        model_deberta_name,
        num_labels=5,
        use_safetensors=True
    )
    model_deberta.eval()
    
    # 2. Load RoBERTa
    model_roberta_name = "roberta-base"
    print(f"Loading {model_roberta_name}...")
    tokenizer_roberta = AutoTokenizer.from_pretrained(model_roberta_name)
    model_roberta = AutoModelForSequenceClassification.from_pretrained(
        model_roberta_name,
        num_labels=5,
        use_safetensors=True
    )
    model_roberta.eval()
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_deberta = model_deberta.to(device)
    model_roberta = model_roberta.to(device)
    
    positive_gain_count = 0
    
    # Inference loop
    for idx in tqdm(range(len(test_df)), desc="Calculating Confidence Gain"):
        prompt = prompts[idx]
        
        # DeBERTa inference
        inputs_deberta = tokenizer_deberta(prompt, return_tensors="pt", truncation=True, max_length=512)
        inputs_deberta = {k: v.to(device) for k, v in inputs_deberta.items()}
        with torch.no_grad():
            outputs_d = model_deberta(**inputs_deberta)
            probs_deberta = F.softmax(outputs_d.logits, dim=-1).squeeze().cpu().numpy()
            
        # RoBERTa inference
        inputs_roberta = tokenizer_roberta(prompt, return_tensors="pt", truncation=True, max_length=512)
        inputs_roberta = {k: v.to(device) for k, v in inputs_roberta.items()}
        with torch.no_grad():
            outputs_r = model_roberta(**inputs_roberta)
            probs_roberta = F.softmax(outputs_r.logits, dim=-1).squeeze().cpu().numpy()
            
        # Weighted Ensemble calculation (0.7 * DeBERTa + 0.3 * RoBERTa)
        w_deberta = 0.70
        w_roberta = 0.30
        probs_ensemble = (w_deberta * probs_deberta) + (w_roberta * probs_roberta)
        
        # Confidences (highest class probability)
        conf_deberta = np.max(probs_deberta)
        conf_ensemble = np.max(probs_ensemble)
        
        gain = conf_ensemble - conf_deberta
        
        if gain > 0:
            positive_gain_count += 1
            
    print("\n==================================================")
    print(f"Number of rows with positive confidence gain (> 0): {positive_gain_count}")
    print("==================================================")

if __name__ == "__main__":
    main()

Loading dataset: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
Loading microsoft/deberta-v3-small...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         

Loading roberta-base...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Calculating Confidence Gain: 100%|██████████| 100/100 [00:02<00:00, 49.38it/s]



Number of rows with positive confidence gain (> 0): 0


For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:

1. DeBERTa alone

2. Weighted Ensemble

Question 9:

# How many rows have at least one change in their ordered Top-3 ranking after ensembling?

# Examples:

# A C D vs. A D C

In [10]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

def main():
    test_df_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
    print(f"Loading dataset: {test_df_path}")
    test_df = pd.read_csv(test_df_path).head(100)
    
    prompts = test_df['prompt'].astype(str).tolist()
    options = ['A', 'B', 'C', 'D', 'E']
    
    # 1. Load DeBERTa
    model_deberta_name = "microsoft/deberta-v3-small"
    print(f"Loading {model_deberta_name}...")
    tokenizer_deberta = AutoTokenizer.from_pretrained(model_deberta_name)
    model_deberta = AutoModelForSequenceClassification.from_pretrained(
        model_deberta_name,
        num_labels=5,
        use_safetensors=True
    )
    model_deberta.eval()
    
    # 2. Load RoBERTa
    model_roberta_name = "roberta-base"
    print(f"Loading {model_roberta_name}...")
    tokenizer_roberta = AutoTokenizer.from_pretrained(model_roberta_name)
    model_roberta = AutoModelForSequenceClassification.from_pretrained(
        model_roberta_name,
        num_labels=5,
        use_safetensors=True
    )
    model_roberta.eval()
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_deberta = model_deberta.to(device)
    model_roberta = model_roberta.to(device)
    
    differing_top3_count = 0
    
    # Inference loop
    for idx in tqdm(range(len(test_df)), desc="Comparing Top-3 Predictions"):
        prompt = prompts[idx]
        
        # DeBERTa inference
        inputs_deberta = tokenizer_deberta(prompt, return_tensors="pt", truncation=True, max_length=512)
        inputs_deberta = {k: v.to(device) for k, v in inputs_deberta.items()}
        with torch.no_grad():
            outputs_d = model_deberta(**inputs_deberta)
            probs_deberta = F.softmax(outputs_d.logits, dim=-1).squeeze().cpu().numpy()
            
        # RoBERTa inference
        inputs_roberta = tokenizer_roberta(prompt, return_tensors="pt", truncation=True, max_length=512)
        inputs_roberta = {k: v.to(device) for k, v in inputs_roberta.items()}
        with torch.no_grad():
            outputs_r = model_roberta(**inputs_roberta)
            probs_roberta = F.softmax(outputs_r.logits, dim=-1).squeeze().cpu().numpy()
            
        # Weighted Ensemble calculation (0.7 * DeBERTa + 0.3 * RoBERTa)
        w_deberta = 0.70
        w_roberta = 0.30
        probs_ensemble = (w_deberta * probs_deberta) + (w_roberta * probs_roberta)
        
        # Ordered Top-3 lists
        top3_deberta_indices = np.argsort(probs_deberta)[::-1][:3]
        top3_deberta = [options[idx_opt] for idx_opt in top3_deberta_indices]
        
        top3_ensemble_indices = np.argsort(probs_ensemble)[::-1][:3]
        top3_ensemble = [options[idx_opt] for idx_opt in top3_ensemble_indices]
        
        # Compare strings or lists directly (order matters)
        if top3_deberta != top3_ensemble:
            differing_top3_count += 1
            print(f"\nRow {idx} changed Top-3 ranking:")
            print(f"  DeBERTa:  {' '.join(top3_deberta)}")
            print(f"  Ensemble: {' '.join(top3_ensemble)}")
            
    print("\n==================================================")
    print(f"Number of rows with at least one change in their Top-3 ranking: {differing_top3_count}")
    print("==================================================")

if __name__ == "__main__":
    main()

Loading dataset: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
Loading microsoft/deberta-v3-small...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         

Loading roberta-base...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...


Row 73 changed Top-3 ranking:
  DeBERTa:  D E B
  Ensemble: E D B

Row 74 changed Top-3 ranking:
  DeBERTa:  E B D
  Ensemble: E B C

Row 77 changed Top-3 ranking:
  DeBERTa:  D E B
  Ensemble: E D B

Row 78 changed Top-3 ranking:
  DeBERTa:  E D B
  Ensemble: E B D

Row 79 changed Top-3 ranking:
  DeBERTa:  E D B
  Ensemble: E B D

Row 81 changed Top-3 ranking:
  DeBERTa:  D E B
  Ensemble: E D B

Row 82 changed Top-3 ranking:
  DeBERTa:  E B D
  Ensemble: E B C


Comparing Top-3 Predictions:  89%|████████▉ | 89/100 [00:01<00:00, 48.89it/s]


Row 83 changed Top-3 ranking:
  DeBERTa:  B D C
  Ensemble: B C D

Row 87 changed Top-3 ranking:
  DeBERTa:  B D E
  Ensemble: B E D

Row 88 changed Top-3 ranking:
  DeBERTa:  E B C
  Ensemble: E C B

Row 89 changed Top-3 ranking:
  DeBERTa:  E D B
  Ensemble: E B D

Row 91 changed Top-3 ranking:
  DeBERTa:  E D B
  Ensemble: E B D

Row 92 changed Top-3 ranking:
  DeBERTa:  A D C
  Ensemble: A C D


Comparing Top-3 Predictions: 100%|██████████| 100/100 [00:02<00:00, 48.30it/s]



Row 95 changed Top-3 ranking:
  DeBERTa:  D E B
  Ensemble: E D B

Row 97 changed Top-3 ranking:
  DeBERTa:  D B C
  Ensemble: B D C

Row 98 changed Top-3 ranking:
  DeBERTa:  E D B
  Ensemble: E B D

Number of rows with at least one change in their Top-3 ranking: 51


Using the Top-3 predictions generated by your weighted ensemble for the first 100 validation samples, compute the MAP@3 score.

Question 10:


# What is the final MAP@3 score? 

(Round to 4 decimal places.)


In [12]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

def map3(predictions, targets):
    """
    Computes the Mean Average Precision at 3 (MAP@3).
    """
    scores = []
    for pred, target in zip(predictions, targets):
        score = 0.0
        for i, p in enumerate(pred[:3]):
            if p == target:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return float(np.mean(scores)) if scores else 0.0

def main():
    # Load first 100 rows of train.csv (which contain ground truth labels)
    train_df_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
    print(f"Loading dataset: {train_df_path}")
    train_df = pd.read_csv(train_df_path).head(100)
    
    prompts = train_df['prompt'].astype(str).tolist()
    targets = train_df['answer'].tolist()
    options = ['A', 'B', 'C', 'D', 'E']
    
    # 1. Load DeBERTa
    model_deberta_name = "microsoft/deberta-v3-small"
    print(f"Loading {model_deberta_name}...")
    tokenizer_deberta = AutoTokenizer.from_pretrained(model_deberta_name)
    model_deberta = AutoModelForSequenceClassification.from_pretrained(
        model_deberta_name,
        num_labels=5,
        use_safetensors=True
    )
    model_deberta.eval()
    
    # 2. Load RoBERTa
    model_roberta_name = "roberta-base"
    print(f"Loading {model_roberta_name}...")
    tokenizer_roberta = AutoTokenizer.from_pretrained(model_roberta_name)
    model_roberta = AutoModelForSequenceClassification.from_pretrained(
        model_roberta_name,
        num_labels=5,
        use_safetensors=True
    )
    model_roberta.eval()
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_deberta = model_deberta.to(device)
    model_roberta = model_roberta.to(device)
    
    predictions = []
    
    # Inference loop
    for idx in tqdm(range(len(train_df)), desc="Calculating MAP@3"):
        prompt = prompts[idx]
        
        # DeBERTa inference
        inputs_deberta = tokenizer_deberta(prompt, return_tensors="pt", truncation=True, max_length=512)
        inputs_deberta = {k: v.to(device) for k, v in inputs_deberta.items()}
        with torch.no_grad():
            outputs_d = model_deberta(**inputs_deberta)
            probs_deberta = F.softmax(outputs_d.logits, dim=-1).squeeze().cpu().numpy()
            
        # RoBERTa inference
        inputs_roberta = tokenizer_roberta(prompt, return_tensors="pt", truncation=True, max_length=512)
        inputs_roberta = {k: v.to(device) for k, v in inputs_roberta.items()}
        with torch.no_grad():
            outputs_r = model_roberta(**inputs_roberta)
            probs_roberta = F.softmax(outputs_r.logits, dim=-1).squeeze().cpu().numpy()
            
        # Weighted Ensemble calculation (0.7 * DeBERTa + 0.3 * RoBERTa)
        w_deberta = 0.70
        w_roberta = 0.30
        probs_ensemble = (w_deberta * probs_deberta) + (w_roberta * probs_roberta)
        
        # Top-3 predictions
        sorted_indices = np.argsort(probs_ensemble)[::-1]
        top_3 = [options[idx_opt] for idx_opt in sorted_indices[:3]]
        predictions.append(top_3)
        
    # Calculate MAP@3
    map3_score = map3(predictions, targets)
    print("\n==================================================")
    print(f"Final MAP@3 score on first 100 samples: {map3_score:.4f}")
    print("==================================================")

if __name__ == "__main__":
    main()

Loading dataset: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
Loading microsoft/deberta-v3-small...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         

Loading roberta-base...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Calculating MAP@3: 100%|██████████| 100/100 [00:02<00:00, 49.38it/s]



Final MAP@3 score on first 100 samples: 0.4467
